# Feature matching between campod stills, against the FC's motion

For one campod session: SIFT keypoints on every still, ratio-test matches between stills `k` apart,
and a RANSAC essential matrix on each pair. Each pair's recovered relative rotation is set against
the rotation the FC's attitude estimate says happened between the two exposures.

**What this measures.** For each pair:
- how many keypoints each frame has;
- how many matches survive the ratio test;
- how many of those survive RANSAC.

For the pairs RANSAC accepts, it measures:
- whether the camera's rotation agrees with the FC's;
- whether the direction the camera moved agrees with the EKF's displacement.

**What it is not.** No bundle adjustment, no tracks longer than a pair, no map. Pairs only.

**The lens is an assumption.** The B0310 is recorded only as a fixed M12 lens with a 120 deg
horizontal field. No calibration exists. Pixels are turned into rays with a radially symmetric
model (`lens_model`, default equidistant, `r = f*theta`), with `f` set so the image width spans
`hfov_deg`, and the principal point at the image centre.

The rotation **angle** between two camera poses does not depend on how the camera is mounted, so
it compares with the FC's angle directly. A wrong lens model shows up as camera angles that scale
away from the FC's; the fitted slope is reported. Comparing the rotation **axis** and the
translation direction needs the camera-to-body rotation. That is fitted from the pairs themselves
and reported with its residual. It is not a measured mount.

Stills are placed on the FC clock by `exposure_wall_ns` through the FC's GPS->UTC fit, as in
`image-sharpness-vs-motion.ipynb`.

**Memory.** One SIFT worker peaks at about 0.9 GB on a 1536x864 still (measured, skimage 0.25),
and finds about 16k keypoints. Matching runs in row blocks and peaks at about 0.4 GB per pair;
skimage's `match_descriptors` builds the whole distance matrix, about 3 GB for two such stills. `n_jobs` is a small fixed number on purpose: the pod this runs in is
shared, and sizing the pool to the CPU count there OOM-kills it.

In [ ]:
# --- papermill parameters ---
flight_dir = "/home/jovyan/datasets/flights/rekon10/2026-10-27-ffocus"
captures_dir = ""   # "" -> <flight_dir>/captures
out_dir = ""        # "" -> <flight_dir>/derived
fc_bin = ""         # "" -> the one *.bin at the flight root, or under fc/
campod = ""         # "" -> every campod session found; or e.g. "campod-se"
strides = [1, 2, 3, 5, 10]   # pair each still with the one this many stills later
downscale = 3       # SIFT runs on the still shrunk by this factor (4608x2592 -> 1536x864 at 3)
ratio = 0.8         # Lowe ratio test
ransac_px = 1.5     # RANSAC inlier threshold, in pixels at the downscaled size
lens_model = "equidistant"   # "equidistant" (r = f*theta) or "pinhole" (r = f*tan(theta))
hfov_deg = 120.0    # horizontal field of view the lens model is fitted to
min_inliers = 30    # a pair with fewer RANSAC inliers is reported as failed, not as a pose
n_jobs = 4          # worker processes for SIFT and matching

In [ ]:
import os, sys, json, hashlib, subprocess, datetime as dt
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy.spatial.transform import Rotation as Rot, Slerp
from skimage.feature import SIFT
from skimage.measure import ransac
from skimage.transform import EssentialMatrixTransform

sys.path.insert(0, str(Path.cwd() / "analysis") if (Path.cwd() / "analysis").is_dir() else str(Path.cwd()))
from ardupilot_log import parse_log

D = Path(flight_dir)
CAP = Path(captures_dir) if captures_dir else D / "captures"
OUT = Path(out_dir) if out_dir else D / "derived"
OUT.mkdir(parents=True, exist_ok=True)
if fc_bin:
    FC_BIN = Path(fc_bin)
else:
    bins = sorted(D.glob("*.bin")) + sorted((D / "fc").glob("*.bin"))
    assert len(bins) == 1, f"name fc_bin: {len(bins)} .bin files under {D}"
    FC_BIN = bins[0]

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

try:
    git_sha = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
except Exception:
    git_sha = None

sessions = []
for node_dir in sorted(CAP.glob("campod-*")):
    if campod and node_dir.name != campod:
        continue
    for s in sorted(p for p in node_dir.iterdir() if p.is_dir()):
        jpgs = sorted(s.glob("*.jpg")) + sorted((s / "stills").glob("*.jpg"))
        if jpgs:
            sessions.append((node_dir.name, s, jpgs))
assert sessions, f"no campod session with stills under {CAP}"
print("FC log:", FC_BIN.name)
for n, s, j in sessions:
    print(f"  {n}/{s.name}: {len(j)} stills")

## The FC: arm, liftoff, the GPS->UTC fit, attitude and EKF position

In [ ]:
F, fc_dur, parms = parse_log(FC_BIN, ["ARM", "ATT", "XKF1", "GPS"])
A = F["ARM"]
T_ARM = float(A[A.ArmState == 1].t_s.iloc[0]); T_DIS = float(A[A.ArmState == 0].t_s.iloc[-1])
xk = F["XKF1"]; xk = xk[xk.C == 0].sort_values("t_s").drop_duplicates("t_s")
alt = -xk.PD.values
ground = float(np.median(alt[(xk.t_s.values > T_ARM - 30) & (xk.t_s.values < T_ARM)]))
up = np.where((xk.t_s.values > T_ARM) & (alt > ground + 0.30))[0]
T_LIFT = float(xk.t_s.values[up[0]]) if len(up) else T_ARM
down = np.where((xk.t_s.values < T_DIS) & (alt > ground + 0.30))[0]
T_TOUCH = float(xk.t_s.values[down[-1]]) if len(down) else T_DIS

GPS_EPOCH = dt.datetime(1980, 1, 6, tzinfo=dt.timezone.utc); LEAP = 18
g = F["GPS"]; g = g[(g.I == 0) & (g.GWk > 1000)].sort_values("t_s")
assert len(g) >= 10, "campod stills are placed through the FC's GPS->UTC fit, and this log has no GPS week"
utc = np.array([(GPS_EPOCH + dt.timedelta(weeks=int(w), milliseconds=int(ms))).timestamp() - LEAP
                for w, ms in zip(g.GWk.values, g.GMS.values)])
sl, ic = np.polyfit(g.t_s.values, utc, 1)
gps_fit = {"rows": len(g), "residual_ms": round(float((utc - (ic + sl * g.t_s.values)).std() * 1000), 1)}
utc_to_fc = lambda u: (np.asarray(u) - ic) / sl

att = F["ATT"].sort_values("t_s").drop_duplicates("t_s")
# ArduPilot ATT is body attitude in NED, ZYX Euler (yaw, pitch, roll), degrees
R_att = Rot.from_euler("ZYX", np.c_[att.Yaw.values, att.Pitch.values, att.Roll.values], degrees=True)
slerp = Slerp(att.t_s.values, R_att)
def body_R(t):
    return slerp(np.clip(np.asarray(t, float), att.t_s.values[0], att.t_s.values[-1]))
def ned(t):
    return np.array([np.interp(t, xk.t_s.values, xk[c].values) for c in ("PN", "PE", "PD")])

air = (xk.t_s.values > T_LIFT) & (xk.t_s.values < T_TOUCH)
print(f"ARM {T_ARM:.1f} s, liftoff {T_LIFT:.1f} s, touchdown {T_TOUCH:.1f} s, DISARM {T_DIS:.1f} s; "
      f"height above takeoff in flight p10/50/90 {np.round(np.percentile(alt[air] - ground, [10, 50, 90]), 1).tolist()} m")
print("FC->UTC fit:", gps_fit, f"; ATT {len(att)} rows ({len(att) / (att.t_s.iloc[-1] - att.t_s.iloc[0]):.0f} Hz)")

## Keypoints on every still

SIFT on the grey still, shrunk by `downscale`. Each keypoint is turned into a ray with the lens
model, kept as normalised image coordinates (x/z, y/z) for the essential matrix.

Keypoints and descriptors are cached per still in `<out_dir>/feature-cache/<pod>/<still>.npz`,
tagged with the skimage version and the size SIFT ran at. A still whose cache entry carries the
same tag is not recomputed. Every pair's numbers are written to `feature-matching-pairs.csv`.

In [ ]:
W_FULL, H_FULL = Image.open(sessions[0][2][0]).size
W, H = W_FULL // downscale, H_FULL // downscale
cx, cy = (W - 1) / 2.0, (H - 1) / 2.0
half = np.radians(hfov_deg / 2)
if lens_model == "equidistant":
    f_px = (W / 2) / half
    pix_to_theta = lambda r: r / f_px
elif lens_model == "pinhole":
    f_px = (W / 2) / np.tan(half)
    pix_to_theta = lambda r: np.arctan(r / f_px)
else:
    raise ValueError(lens_model)

def to_normalised(rc):
    """(row, col) keypoints at the downscaled size -> normalised image coords (x/z, y/z)."""
    x = rc[:, 1] - cx; y = rc[:, 0] - cy
    r = np.hypot(x, y); th = pix_to_theta(r)
    s = np.where(r > 0, np.tan(th) / np.maximum(r, 1e-9), 1.0 / f_px)
    return np.c_[x * s, y * s]

import skimage
CACHE = OUT / "feature-cache"
CACHE_TAG = f"sift skimage {skimage.__version__} {W}x{H}"

def sift_one(args):
    """Keypoints and descriptors for one still, read from the cache when it was made the same way."""
    path, node = args
    cpath = CACHE / node / (Path(path).stem + ".npz")
    if cpath.exists():
        z = np.load(cpath)
        if str(z["tag"]) == CACHE_TAG:
            return z["keypoints"], z["descriptors"], True
    im = Image.open(path).convert("L").resize((W, H), Image.BILINEAR)
    det = SIFT()
    det.detect_and_extract(np.asarray(im, dtype=np.float32) / 255.0)
    kp, desc = det.keypoints.astype(np.float32), det.descriptors
    cpath.parent.mkdir(parents=True, exist_ok=True)
    tmp = cpath.with_suffix(".tmp.npz")
    np.savez(tmp, keypoints=kp, descriptors=desc, tag=CACHE_TAG)
    os.replace(tmp, cpath)
    return kp, desc, False

print(f"{W_FULL}x{H_FULL} -> {W}x{H}; {lens_model} f = {f_px:.1f} px at that size; "
      f"corner ray at {np.degrees(pix_to_theta(np.hypot(W / 2, H / 2))):.1f} deg off axis")

FR = {}
for node, sdir, jpgs in sessions:
    side = [json.load(open(str(p)[:-4] + ".json")) for p in jpgs]
    t_fc = utc_to_fc([s["exposure_wall_ns"] / 1e9 for s in side])
    with ProcessPoolExecutor(n_jobs) as ex:
        got = list(ex.map(sift_one, [(str(p), node) for p in jpgs], chunksize=2))
    feats = [(k, d) for k, d, _ in got]
    print(f"{node}: {sum(c for _, _, c in got)} of {len(got)} stills' features read from {CACHE / node}")
    df = pd.DataFrame(dict(name=[p.name for p in jpgs], seq=[s.get("seq") for s in side], t_fc=t_fc,
                           n_kp=[len(k) for k, _ in feats]))
    df["airborne"] = (df.t_fc > T_LIFT + 2) & (df.t_fc < T_TOUCH - 2)
    FR[node] = dict(df=df, feats=feats, jpgs=jpgs)
    a = df[df.airborne]
    print(f"{node}: {len(df)} stills, {len(a)} airborne; keypoints per airborne still "
          f"p10/50/90 {np.percentile(a.n_kp, [10, 50, 90]).astype(int).tolist()}")

## Matching and RANSAC, every pair `k` stills apart

Matches pass the ratio test and must also be mutual nearest neighbours. RANSAC fits an essential
matrix on the normalised coordinates. The residual is the Sampson distance, with the threshold set
from `ransac_px` divided by `f`. The pose is the one of the four `E` decompositions that puts the
most inliers in front of both cameras. That pose is the camera's relative rotation and its unit
translation; the translation's scale is not observable from a pair.

The FC side of each pair:
- the relative rotation between the two exposure times, from `ATT`;
- the EKF displacement, from `XKF1`, in the first frame's body axes.

In [ ]:
from scipy.spatial.distance import cdist

def match_chunked(d1, d2, max_ratio=0.8, chunk=1000):
    """skimage.feature.match_descriptors(d1, d2, cross_check=True, max_ratio=max_ratio) with the
    euclidean metric, computed a block of rows at a time so the full distance matrix never exists
    (about 3 GB for two 20k-keypoint stills). cdist on a block gives the same numbers as the full
    cdist, and ties resolve to the first index as argmin does; identical output was checked on
    three pairs from 2026-10-27-ffocus."""
    n1, n2 = len(d1), len(d2)
    idx2 = np.empty(n1, int); best = np.empty(n1); second = np.empty(n1)
    col_min = np.full(n2, np.inf); col_arg = np.zeros(n2, int)
    for s in range(0, n1, chunk):
        Dm = cdist(d1[s:s + chunk], d2)
        r = np.arange(len(Dm)); j = Dm.argmin(axis=1)
        idx2[s:s + len(Dm)] = j; b = Dm[r, j]; best[s:s + len(Dm)] = b
        cm = Dm.min(axis=0); ca = Dm.argmin(axis=0) + s
        upd = cm < col_min; col_min[upd] = cm[upd]; col_arg[upd] = ca[upd]
        Dm[r, j] = np.inf; second[s:s + len(Dm)] = Dm.min(axis=1)
    i1 = np.arange(n1)
    keep = col_arg[idx2] == i1
    i1, j, b, sb = i1[keep], idx2[keep], best[keep], second[keep]
    if max_ratio < 1.0:
        sb = np.where(sb == 0, np.finfo(np.float64).eps, sb)
        keep = (b / sb) < max_ratio
        i1, j = i1[keep], j[keep]
    return np.column_stack((i1, j))

def decompose(E, x1, x2):
    """The (R, t) of the four E decompositions with the most points in front of both cameras,
    for x2 ~ R x1 + t. Depths by least squares on d2*p2 = d1*R*p1 + t."""
    U, _, Vt = np.linalg.svd(E)
    if np.linalg.det(U) < 0: U = -U
    if np.linalg.det(Vt) < 0: Vt = -Vt
    Wm = np.array([[0, -1, 0], [1, 0, 0], [0, 0, 1]])
    p1 = np.c_[x1, np.ones(len(x1))]; p2 = np.c_[x2, np.ones(len(x2))]
    best = None
    for R in (U @ Wm @ Vt, U @ Wm.T @ Vt):
        for t in (U[:, 2], -U[:, 2]):
            A = np.stack([p1 @ R.T, -p2], axis=2)                  # n x 3 x 2
            AtA = np.einsum("nki,nkj->nij", A, A)
            Atb = np.einsum("nki,k->ni", A, -t)
            d = np.linalg.solve(AtA + 1e-12 * np.eye(2), Atb[..., None])[..., 0]
            good = int(((d[:, 0] > 0) & (d[:, 1] > 0)).sum())
            if best is None or good > best[0]:
                best = (good, R, t)
    return best

def match_pair(args):
    node, i, j = args
    (k1, d1), (k2, d2) = FR[node]["feats"][i], FR[node]["feats"][j]
    if len(k1) < 8 or len(k2) < 8:
        return dict(matches=0, inliers=0)
    m = match_chunked(d1, d2, max_ratio=ratio)
    out = dict(matches=len(m), inliers=0)
    if len(m) < 8:
        return out
    x1 = to_normalised(k1[m[:, 0]]); x2 = to_normalised(k2[m[:, 1]])
    model, inl = ransac((x1, x2), EssentialMatrixTransform, min_samples=8,
                        residual_threshold=ransac_px / f_px, max_trials=2000, rng=0)
    if model is None or inl is None:
        return out
    out["inliers"] = int(inl.sum())
    if inl.sum() >= min_inliers:
        good, R, t = decompose(model.params, x1[inl], x2[inl])
        out.update(front_frac=good / inl.sum(), R=R, t=t,
                   flow_px=float(np.median(np.hypot(*(k2[m[inl, 1]] - k1[m[inl, 0]]).T))))
    return out

rows = []
for node, fr in FR.items():
    df = fr["df"]; idx = np.where(df.airborne.values)[0]; idx_set = set(idx.tolist())
    jobs = [(node, int(i), int(i + k)) for k in strides for i in idx if int(i + k) in idx_set]
    with ProcessPoolExecutor(n_jobs) as ex:
        res = list(ex.map(match_pair, jobs, chunksize=4))
    for (nd, i, j), r in zip(jobs, res):
        ti, tj = df.t_fc.values[i], df.t_fc.values[j]
        Rb = body_R([ti, tj]); Ri, Rj = Rb[0], Rb[1]
        R_rel_body = Ri.inv() * Rj                        # body j expressed in body i
        disp_body = Ri.apply(ned(tj) - ned(ti), inverse=True)
        row = dict(camera=node, i=i, j=j, stride=j - i, dt_s=tj - ti, t_fc=ti,
                   kp_i=df.n_kp.values[i], kp_j=df.n_kp.values[j], matches=r["matches"], inliers=r["inliers"],
                   fc_angle_deg=float(np.degrees(R_rel_body.magnitude())),
                   fc_rotvec=R_rel_body.as_rotvec(), disp_body=disp_body,
                   disp_m=float(np.linalg.norm(disp_body)))
        if "R" in r:
            # E maps camera i to camera j: x_j = R x_i + t. The camera's own rotation from i to j is R^T,
            # and camera j's centre in camera i's axes is -R^T t.
            Rc = Rot.from_matrix(r["R"]).inv()
            row.update(cam_angle_deg=float(np.degrees(Rc.magnitude())), cam_rotvec=Rc.as_rotvec(),
                       cam_t=-r["R"].T @ r["t"], front_frac=r["front_frac"], flow_px=r["flow_px"])
        rows.append(row)
P = pd.DataFrame(rows)
P["posed"] = P.inliers >= min_inliers
# every pair's numbers, vectors split into columns, so they can be re-read without recomputing
flat = P.drop(columns=["fc_rotvec", "disp_body", "cam_rotvec", "cam_t"], errors="ignore").copy()
for col in ("fc_rotvec", "disp_body", "cam_rotvec", "cam_t"):
    if col in P:
        v = np.stack([x if isinstance(x, np.ndarray) else np.full(3, np.nan) for x in P[col]])
        for k, ax_ in enumerate("xyz"):
            flat[f"{col}_{ax_}"] = v[:, k]
flat.to_csv(OUT / "feature-matching-pairs.csv", index=False)
print(P.groupby(["camera", "stride"]).agg(pairs=("i", "size"), dt_s=("dt_s", "median"),
      matches_p50=("matches", "median"), inliers_p10=("inliers", lambda v: int(np.percentile(v, 10))),
      inliers_p50=("inliers", "median"), posed=("posed", "sum"),
      fc_angle_p50=("fc_angle_deg", "median"), disp_m_p50=("disp_m", "median")).to_string())

## Rotation: camera against FC

The rotation angle is compared directly; it does not depend on how the camera is mounted. The
slope of camera angle on FC angle is the check on the lens model: an `f` that is too long or too
short scales every recovered angle.

The camera-to-body rotation `C` is fitted from the pairs. It is the rotation that best maps camera
rotation vectors onto FC rotation vectors (Kabsch, weighted by angle, pairs over 1 deg). The
residual axis angle is how far each pair's axis sits from that common mount.

In [ ]:
def kabsch(a, b, w):
    """Rotation C minimising sum w |C a - b|^2."""
    Hm = (a * w[:, None]).T @ b
    U, _, Vt = np.linalg.svd(Hm)
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    return Vt.T @ np.diag([1, 1, d]) @ U.T

ROT = {}
fig, axes = plt.subplots(1, len(FR), figsize=(6 * len(FR), 5), squeeze=False)
for ax, node in zip(axes[0], FR):
    q = P[(P.camera == node) & P.posed].copy()
    x, y = q.fc_angle_deg.values, q.cam_angle_deg.values
    slope = float(np.sum(x * y) / np.sum(x * x)) if len(q) else np.nan
    err = y - x
    use = q[q.fc_angle_deg > 1.0]
    C = None; ax_err = np.array([])
    if len(use) >= 5:
        a = np.stack(use.cam_rotvec.values); b = np.stack(use.fc_rotvec.values)
        C = kabsch(a, b, np.linalg.norm(b, axis=1))
        ca = a @ C.T
        cosang = np.sum(ca * b, axis=1) / (np.linalg.norm(ca, axis=1) * np.linalg.norm(b, axis=1))
        ax_err = np.degrees(np.arccos(np.clip(cosang, -1, 1)))
    ROT[node] = dict(n=len(q), slope=slope, C=C, ax_err=ax_err, angle_err=err)
    for k in sorted(q.stride.unique()):
        s = q[q.stride == k]
        ax.scatter(s.fc_angle_deg, s.cam_angle_deg, s=8, label=f"{k} apart")
    lim = max(1.0, np.nanmax(np.r_[x, y]) if len(q) else 1.0)
    ax.plot([0, lim], [0, lim], "k--", lw=0.8, label="equal")
    ax.set_xlabel("FC rotation between exposures (deg)"); ax.set_ylabel("camera rotation from E (deg)")
    ax.set_title(f"{node}: {len(q)} posed pairs, slope {slope:.3f}"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    print(f"{node}: angle error cam-FC (deg) p10/50/90 {np.round(np.percentile(err, [10, 50, 90]), 2).tolist() if len(q) else None}; "
          f"slope {slope:.3f}; fitted mount from {len(use)} pairs, axis error (deg) p50/90 "
          f"{np.round(np.percentile(ax_err, [50, 90]), 1).tolist() if len(ax_err) else None}")
fig.tight_layout(); fig.savefig(OUT / "feature-matching-rotation.png", dpi=110); plt.show()

## Translation direction: camera against EKF

Only for pairs where the EKF moved at least 0.3 m between the exposures; below that the direction
is noise. The camera's unit translation is rotated into body axes by the fitted mount `C`, then set
against the EKF displacement in the first frame's body axes.

In [ ]:
TR = {}
for node in FR:
    q = P[(P.camera == node) & P.posed & (P.disp_m >= 0.3)]
    C = ROT[node]["C"]
    if C is None or not len(q):
        TR[node] = dict(n=0, err=np.array([])); print(f"{node}: no pairs to compare"); continue
    tc = np.stack(q.cam_t.values) @ C.T; tb = np.stack(q.disp_body.values)
    cosang = np.sum(tc * tb, axis=1) / (np.linalg.norm(tc, axis=1) * np.linalg.norm(tb, axis=1))
    err = np.degrees(np.arccos(np.clip(cosang, -1, 1)))
    TR[node] = dict(n=len(q), err=err)
    print(f"{node}: {len(q)} pairs moved >= 0.3 m; direction error (deg) p10/50/90 "
          f"{np.round(np.percentile(err, [10, 50, 90]), 1).tolist()}; under 15 deg: {int((err < 15).sum())}")

## Counts along the flight, and one pair drawn

In [ ]:
fig, axes = plt.subplots(len(FR), 1, figsize=(13, 3.2 * len(FR)), squeeze=False, sharex=True)
for ax, node in zip(axes[:, 0], FR):
    q = P[(P.camera == node) & (P.stride == min(strides))]
    ax.plot(q.t_fc, q.kp_i, lw=0.8, label="keypoints (first still)")
    ax.plot(q.t_fc, q.matches, lw=0.8, label="ratio-test matches")
    ax.plot(q.t_fc, q.inliers, lw=1.2, label="RANSAC inliers")
    ax.axhline(min_inliers, color="k", ls=":", lw=0.8)
    ax.set_yscale("log"); ax.set_ylabel(node); ax.grid(alpha=0.3); ax.legend(fontsize=8, loc="lower left")
axes[-1, 0].set_xlabel(f"FC time (s), pairs {min(strides)} still(s) apart")
fig.tight_layout(); fig.savefig(OUT / "feature-matching-counts.png", dpi=110); plt.show()

node = next(iter(FR)); q = P[(P.camera == node) & (P.stride == min(strides)) & P.posed]
if len(q):
    r = q.iloc[(q.inliers - q.inliers.median()).abs().argsort().values[0]]
    i, j = int(r.i), int(r.j)
    (k1, d1), (k2, d2) = FR[node]["feats"][i], FR[node]["feats"][j]
    m = match_chunked(d1, d2, max_ratio=ratio)
    x1 = to_normalised(k1[m[:, 0]]); x2 = to_normalised(k2[m[:, 1]])
    _, inl = ransac((x1, x2), EssentialMatrixTransform, min_samples=8,
                    residual_threshold=ransac_px / f_px, max_trials=2000, rng=0)
    im1 = np.asarray(Image.open(FR[node]["jpgs"][i]).resize((W, H)))
    fig, ax = plt.subplots(figsize=(13, 13 * H / W))
    ax.imshow(im1)
    for a_, b_, ok in zip(k1[m[:, 0]], k2[m[:, 1]], inl):
        ax.plot([a_[1], b_[1]], [a_[0], b_[0]], color="lime" if ok else "red", lw=0.6, alpha=0.8 if ok else 0.4)
    ax.set_title(f"{node} {FR[node]['df'].name[i]}: {int(inl.sum())} inliers (green) of {len(m)} matches; "
                 f"lines run to the matched point in the next still"); ax.axis("off")
    fig.tight_layout(); fig.savefig(OUT / "feature-matching-example.png", dpi=100); plt.show()

In [ ]:
def pct(v, qs=(10, 50, 90)):
    v = np.asarray(v, float)
    return [round(float(x), 2) for x in np.percentile(v, qs)] if len(v) else None

summary = {"provenance": {"flight": D.name, "notebook": "analysis/feature-matching.ipynb", "coordinator_git_sha": git_sha,
                          "fc_bin": {"name": FC_BIN.name, "sha256": sha256(FC_BIN)}, "captures_dir": str(CAP)},
           "method": {"detector": "skimage SIFT", "downscale": downscale, "size": [W, H], "ratio": ratio,
                      "cross_check": True, "ransac": "EssentialMatrixTransform, Sampson", "ransac_px": ransac_px,
                      "min_inliers": min_inliers, "lens_model": lens_model, "hfov_deg": hfov_deg,
                      "f_px": round(f_px, 2), "lens_is_assumed": True},
           "fc": {"t_liftoff": round(T_LIFT, 2), "t_touchdown": round(T_TOUCH, 2), "gps_utc_fit": gps_fit},
           "cameras": {}}
for node, fr in FR.items():
    df = fr["df"]; a = df[df.airborne]; q = P[P.camera == node]
    by = {}
    for k in strides:
        s = q[q.stride == k]
        by[str(k)] = {"pairs": len(s), "dt_s_median": round(float(s.dt_s.median()), 2) if len(s) else None,
                      "matches": pct(s.matches), "inliers": pct(s.inliers),
                      "inlier_frac_of_matches": pct(s.inliers / s.matches.clip(lower=1)),
                      "posed": int(s.posed.sum()), "fc_angle_deg": pct(s.fc_angle_deg), "disp_m": pct(s.disp_m),
                      "flow_px": pct(s[s.posed].flow_px)}
    R_ = ROT[node]; T_ = TR[node]
    summary["cameras"][node] = {
        "stills": len(df), "airborne": len(a), "keypoints_airborne": pct(a.n_kp),
        "by_stride": by,
        "rotation": {"posed_pairs": R_["n"], "angle_err_deg": pct(R_["angle_err"]), "slope_cam_on_fc": round(R_["slope"], 4),
                     "fitted_mount_C": R_["C"].round(4).tolist() if R_["C"] is not None else None,
                     "axis_err_deg": pct(R_["ax_err"])},
        "translation_direction": {"pairs_moved_0p3m": T_["n"], "err_deg": pct(T_["err"]),
                                  "under_15_deg": int((T_["err"] < 15).sum()) if T_["n"] else 0}}
(OUT / "feature-matching.json").write_text(json.dumps(summary, indent=1) + "\n")
print(json.dumps(summary["cameras"], indent=1))